In [1]:
!pip install pandera

import json
import boto3
import pandas as pd
import numpy as np
import pandera as pa
from pandera import Column, DataFrameSchema, Check

EXPECTED_DIM = 1024

# ---------- rules ----------
def clean_meta(m: dict) -> dict:
    if not isinstance(m, dict):
        return {}
    out = {}
    for k, v in m.items():
        if v is None:
            continue
        if isinstance(v, (str, int, float, bool)):
            out[k] = v
        elif isinstance(v, list) and all(isinstance(x, str) for x in v):
            out[k] = v
    return out

def is_valid_vector(v) -> bool:
    if v is None:
        return False
    if isinstance(v, np.ndarray):
        arr = v
    elif isinstance(v, (list, tuple)):
        arr = np.asarray(v)
    else:
        return False
    if arr.ndim != 1 or arr.shape[0] != EXPECTED_DIM:
        return False
    if arr.dtype == object:
        return False
    if not np.isfinite(arr).all():
        return False
    return True

def metadata_is_pinecone_safe(v) -> bool:
    if v is None or not isinstance(v, str) or v.strip() == "":
        return False
    try:
        obj = json.loads(v)
    except Exception:
        return False
    if not isinstance(obj, dict):
        return False
    return clean_meta(obj) != {}

schema = DataFrameSchema(
    {
        "id": Column(str),
        "values": Column(object, checks=Check(is_valid_vector, element_wise=True)),
        "metadata": Column(str, checks=Check(metadata_is_pinecone_safe, element_wise=True)),
    },
    strict=True,  # no extra columns required? set False if you allow extras
)

# ---------- S3 listing ----------
def list_parquet_keys(bucket: str, prefix: str):
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.lower().endswith(".parquet"):
                yield key

# ---------- bulk validate ----------
def validate_s3_prefix(bucket: str, prefix: str, limit: int | None = None):
    keys = list(list_parquet_keys(bucket, prefix))
    if limit is not None:
        keys = keys[:limit]

    total = len(keys)
    ok = 0
    bad = 0
    failures = {}  # reason -> count

    for i, key in enumerate(keys, 1):
        uri = f"s3://{bucket}/{key}"
        try:
            df = pd.read_parquet(uri)

            # If you expect exactly these 3 columns and want to ignore extras:
            # df = df[["id", "values", "metadata"]]

            schema.validate(df, lazy=False)  # lazy=True => reports all failing rows/checks
            print(f"OK : {uri}")
            ok += 1

        except pa.errors.SchemaError as e:
            #print(f"BAD: {uri}")
            # shorten the message but keep it useful
            msg = str(e).split("failure cases", 1)[0].strip()
            print(f"BAD: {uri} - {msg}")

            failures[msg] = failures.get(msg, 0) + 1
            bad += 1

        except Exception as e:
            #print(f"BAD: {uri}")
            msg = f"Read/parse error: {type(e).__name__}: {e}"
            print(f"BAD: {uri}  - {msg}")

            failures[msg] = failures.get(msg, 0) + 1
            bad += 1

        if i % 50 == 0:
            print(f"...progress: {i}/{total}")

    print("\n====== SUMMARY ======")
    print(f"Total: {total}  OK: {ok}  BAD: {bad}")
    if failures:
        print("\nTop failure reasons:")
        for reason, cnt in sorted(failures.items(), key=lambda x: x[1], reverse=True)[:10]:
            print(f"  - {cnt}x {reason}")

    return {"total": total, "ok": ok, "bad": bad, "failures": failures}

        


/opt/conda/lib/python3.12/site-packages/pandera/_pandas_deprecated.py:146: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


In [2]:
results = validate_s3_prefix(
    bucket="pinecone-s3-bucket-john-ward",
    prefix="pinecone-250k-1024dim/")

OK : s3://pinecone-s3-bucket-john-ward/pinecone-250k-1024dim/ccnews_parquet_fixed-250k/ccnews_vectors_20250718T013638326644.parquet
OK : s3://pinecone-s3-bucket-john-ward/pinecone-250k-1024dim/ccnews_parquet_fixed-250k/ccnews_vectors_20250718T015251197569.parquet
OK : s3://pinecone-s3-bucket-john-ward/pinecone-250k-1024dim/ccnews_parquet_fixed-250k/ccnews_vectors_20250718T020903497512.parquet


KeyboardInterrupt: 